# Lumora Chat Explorer
Explore message balance, conversation intensity, recurring topics, and language signals from a WhatsApp export.


To create the CSV from local ZIP exports, run `python colab/prepare_chat_samples.py` in the Lumora project, then upload `colab/sample_chat_messages.csv` below. The CSV is excluded from Git because it contains private message text.

> **Privacy:** This notebook analyzes message text in your Colab runtime. Upload only chats you have permission to analyze. The prepared CSV pseudonymizes participants and scrubs common phone, email, and URL patterns, but it may still contain identifying details. Review it before sharing.

Emotion and topic charts are keyword estimates. They do not establish anyone?s actual feelings or intent.


In [ ]:
import io
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style='whitegrid', palette=['#7556E8', '#F39B79', '#77BFA3', '#78A9E8'])

try:
    from google.colab import files
    uploaded = files.upload()
    csv_name = next(name for name in uploaded if name.lower().endswith('.csv'))
    df = pd.read_csv(io.BytesIO(uploaded[csv_name]))
except ImportError:
    csv_name = 'sample_chat_messages.csv'
    df = pd.read_csv(csv_name)

required = {'conversation_id', 'timestamp', 'sender', 'text'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Missing required CSV columns: {sorted(missing)}')

df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df['text'] = df['text'].fillna('').astype(str)
df = df.dropna(subset=['timestamp']).sort_values(['conversation_id', 'timestamp']).reset_index(drop=True)
df['message_length'] = df['text'].str.len()
df['hour'] = df['timestamp'].dt.hour
df['weekday'] = df['timestamp'].dt.day_name()
print(f'Loaded {len(df):,} messages from {df.conversation_id.nunique():,} conversations.')


## Conversation snapshot


In [ ]:
first_message = df['timestamp'].min()
last_message = df['timestamp'].max()
summary = pd.DataFrame({
    'Metric': ['Messages', 'Conversations', 'Participants', 'First message', 'Last message'],
    'Value': [f'{len(df):,}', f'{df.conversation_id.nunique():,}', f'{df.sender.nunique():,}',
              first_message.strftime('%d %b %Y'), last_message.strftime('%d %b %Y')]
})
display(summary.style.hide(axis='index').set_properties(**{'text-align':'left'}))

# Show structural fields only; keep message excerpts out of notebook output.
display(df[['conversation_id', 'timestamp', 'sender', 'message_length']].head(10))

## Message balance


In [ ]:
import ipywidgets as widgets
from IPython.display import display as show

conversation_options = ['All conversations'] + sorted(df['conversation_id'].unique().tolist())
conversation_picker = widgets.Dropdown(options=conversation_options, description='Chat:', layout=widgets.Layout(width='320px'))

def selected_rows():
    if conversation_picker.value == 'All conversations':
        return df
    return df[df['conversation_id'] == conversation_picker.value]

def draw_balance(_=None):
    data = selected_rows()
    counts = data['sender'].value_counts()
    fig, ax = plt.subplots(figsize=(6, 4), facecolor='#FBF9FF')
    ax.pie(counts.values, labels=counts.index, autopct='%1.0f%%', startangle=90,
           colors=sns.color_palette(['#7556E8', '#F39B79', '#77BFA3', '#78A9E8'], len(counts)),
           wedgeprops={'width':.42, 'edgecolor':'white', 'linewidth':2})
    ax.set_title('Messages by participant', pad=16, weight='bold')
    plt.show()

conversation_picker.observe(draw_balance, names='value')
show(conversation_picker)
draw_balance()

## Conversation intensity


In [ ]:
period_picker = widgets.ToggleButtons(options=[('Weekly', 'W'), ('Monthly', 'MS')], description='Group by:')

def draw_intensity(_=None):
    data = selected_rows().copy()
    freq = period_picker.value
    data['period'] = data['timestamp'].dt.to_period('W' if freq == 'W' else 'M').dt.start_time
    grouped = data.groupby(['period', 'sender']).size().rename('messages').reset_index()
    fig, ax = plt.subplots(figsize=(12, 4.5), facecolor='#FBF9FF')
    sns.lineplot(data=grouped, x='period', y='messages', hue='sender', marker='o', linewidth=2.5, ax=ax)
    ax.set(title='Messages over time', xlabel='', ylabel='Messages')
    ax.legend(title='Participant', frameon=False, ncol=min(grouped['sender'].nunique(), 4))
    ax.spines[['top', 'right']].set_visible(False)
    plt.xticks(rotation=25, ha='right'); plt.tight_layout(); plt.show()

period_picker.observe(draw_intensity, names='value')
conversation_picker.observe(draw_intensity, names='value')
show(period_picker)
draw_intensity()

## Topic signals
A message can match more than one topic. Counts are simple keyword matches, not semantic classification.


In [ ]:
TOPICS = {
    'Plans & logistics': ['tomorrow', 'today', 'meet', 'where', 'when', 'besok', 'nanti', 'ketemu', 'jadwal'],
    'Campus & classes': ['class', 'lecture', 'course', 'campus', 'dosen', 'kuliah', 'kelas', 'skripsi', 'semester'],
    'Work & projects': ['work', 'project', 'meeting', 'proposal', 'lpj', 'hima', 'event', 'kerja', 'proyek'],
    'Support & advice': ['advice', 'help', 'worry', 'stress', 'capek', 'cerita', 'saran', 'bantu', 'bingung'],
    'Food & everyday life': ['eat', 'food', 'lunch', 'dinner', 'makan', 'minum', 'resto', 'kopi', 'enak'],
}
topic_counts = {topic: int(df['text'].str.contains('|'.join(re.escape(term) for term in terms), case=False, regex=True).sum())
                for topic, terms in TOPICS.items()}
topic_series = pd.Series(topic_counts).sort_values()
fig, ax = plt.subplots(figsize=(9, 4), facecolor='#FBF9FF')
topic_series.plot.barh(ax=ax, color=sns.color_palette('pastel', len(topic_series)))
ax.set(title='Recurring topic keyword signals', xlabel='Messages containing a keyword', ylabel='')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

## Tone keyword signals
These count visible wording and punctuation only. They do not establish a participant?s real emotion or intent.


In [ ]:
TONE = {
    'Excited': ['excited', 'happy', 'seneng', 'seru', 'wkwk', 'haha', '!!'],
    'Angry': ['angry', 'marah', 'kesel', 'kesal', 'sebel', 'annoyed'],
    'Sad': ['sad', 'sedih', 'nangis', 'cry', 'kecewa', 'down'],
}
tone_rows = []
for sender, group in df.groupby('sender'):
    text = group['text'].str.lower()
    for label, terms in TONE.items():
        pattern = '|'.join(re.escape(term) for term in terms)
        share = text.str.contains(pattern, regex=True).mean() * 100
        tone_rows.append({'Participant': sender, 'Signal': label, 'Messages (%)': share})
tone = pd.DataFrame(tone_rows)
fig, ax = plt.subplots(figsize=(10, 4.5), facecolor='#FBF9FF')
sns.barplot(data=tone, x='Participant', y='Messages (%)', hue='Signal', palette=['#F0A078', '#8B69E8', '#83AEE8'], ax=ax)
ax.set(title='Messages with tone keywords', ylabel='Share of messages', xlabel='')
ax.spines[['top', 'right']].set_visible(False); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## When conversations happen


In [ ]:
hourly = df.pivot_table(index='weekday', columns='hour', values='text', aggfunc='count', fill_value=0)
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
hourly = hourly.reindex(weekday_order).fillna(0)
fig, ax = plt.subplots(figsize=(13, 4.5), facecolor='#FBF9FF')
sns.heatmap(hourly, cmap='Purples', linewidths=.4, ax=ax, cbar_kws={'label':'Messages'})
ax.set(title='Message activity by weekday and hour', xlabel='Hour of day', ylabel='')
plt.tight_layout(); plt.show()

## Reading the charts
Use the charts as descriptive summaries. Message volume, keyword matches, and reply timing can show patterns in the export, but they cannot prove relationship quality, intent, or private feelings. Remove identifying details before sharing notebook outputs.
